# Notebook 5 — Forecasting Revenue & Cash

### 📺 Brought to you by **Christian Martinez: AI for Finance**
👉 **Subscribe here: https://www.youtube.com/@christianmartinezAIforFinance**

> *Python for CFOs — a 7-part journey from spreadsheet to strategy.*

---
**Level:** 🟡 Intermediate → 🔴 Advanced  
Stop guessing next quarter. Build three forecasts — moving average, linear trend, and seasonal — then compare their accuracy on real daily sales data.

---

## Dataset: `daily_sales.csv` (2 years of daily sales, included)

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

try:
    ds = pd.read_csv("daily_sales.csv", parse_dates=["date"])
except FileNotFoundError:
    np.random.seed(42)
    days=pd.date_range("2024-01-01","2025-12-31",freq="D"); nd=len(days)
    base=12000+np.linspace(0,4000,nd)
    wk=1-0.3*pd.Series(days.dayofweek).isin([5,6]).astype(int).values
    yr=1+0.1*np.sin(np.arange(nd)/365*2*np.pi)
    ds=pd.DataFrame({"date":days,"sales":(base*wk*yr*(1+np.random.normal(0,.08,nd))).round(0)})
ds = ds.set_index("date")
ds.head()

### 1. Resample to monthly — easier to forecast

In [ ]:
monthly = ds["sales"].resample("MS").sum()
monthly.plot(figsize=(11,4), marker="o", title="Monthly Sales")
plt.ylabel("Sales ($)"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

### 2. Method A — Moving average forecast
Good for stable series. We forecast next month as the average of the last 3.

In [ ]:
window = 3
ma_forecast = monthly.rolling(window).mean().iloc[-1]
print(f"3-month moving average forecast for next month: ${ma_forecast:,.0f}")

### 3. Method B — Linear trend (least-squares regression)
Fit a straight line through time and project it forward.

In [ ]:
x = np.arange(len(monthly))
coef = np.polyfit(x, monthly.values, 1)   # slope, intercept
trend = np.poly1d(coef)

future_x = np.arange(len(monthly), len(monthly)+6)
future_dates = pd.date_range(monthly.index[-1], periods=7, freq="MS")[1:]
forecast = trend(future_x)

print(f"Trend: ${coef[0]:,.0f} growth per month")
for d,v in zip(future_dates, forecast):
    print(f"  {d:%Y-%m}: ${v:,.0f}")

### 4. Plot the trend forecast

In [ ]:
plt.figure(figsize=(11,4))
plt.plot(monthly.index, monthly.values, marker="o", label="Actual")
plt.plot(future_dates, forecast, marker="s", linestyle="--", color="red", label="Forecast")
plt.title("6-Month Revenue Forecast (Linear Trend)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

### 5. Method C — Seasonal forecast
Capture recurring monthly patterns by learning a seasonal factor per calendar month.

In [ ]:
m = monthly.to_frame("sales")
m["month"] = m.index.month
overall_mean = m["sales"].mean()
seasonal_factor = m.groupby("month")["sales"].mean() / overall_mean

# Forecast = trend level * seasonal factor
base_level = trend(future_x)
seasonal_forecast = [lvl * seasonal_factor[d.month] for lvl,d in zip(base_level, future_dates)]

for d,v in zip(future_dates, seasonal_forecast):
    print(f"  {d:%Y-%m}: ${v:,.0f}  (season factor {seasonal_factor[d.month]:.2f})")

### 6. Backtest — which method is most accurate?
Hold out the last 6 months, forecast them, measure error (MAPE).

In [ ]:
train = monthly[:-6]; test = monthly[-6:]
xt = np.arange(len(train))
tcoef = np.polyfit(xt, train.values, 1); ttrend=np.poly1d(tcoef)
pred = ttrend(np.arange(len(train), len(train)+6))

mape = np.mean(np.abs((test.values - pred)/test.values))
print(f"Linear-trend backtest MAPE: {mape:.1%}")
print("(Lower is better — under 10% is solid for monthly finance forecasts.)")

### 🎯 Your turn
Combine trend + seasonality and re-run the backtest. Does MAPE drop? Try a 6-month moving average instead of 3.

---
### ✅ Recap
- Resample noisy daily data to monthly • moving average / trend / seasonal each fit different patterns • **always backtest** with MAPE before trusting a number
- **Next — Notebook 6:** capital allocation — NPV, IRR, and risk.

📺 **Subscribe: https://www.youtube.com/@christianmartinezAIforFinance**